# B2-022 — Session 2: KL Divergence

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

KL divergence measures how badly one distribution $Q$ stands in for another distribution $P$.
It is the regularizer in every VAE loss, and Session 4 derives the VAE objective from an inequality proved here.
This session defines discrete KL with its zero-probability conventions, proves the tangent-line lemma $\log x\le x-1$ and Jensen's inequality for $\log$ from Calculus AB, establishes nonnegativity and asymmetry, contrasts forward and reverse KL, and derives the closed forms for univariate and diagonal Gaussians, ending with the KL to the standard normal used by every VAE.

## 1. Discrete KL and the $0\cdot\log 0$ convention

Let $P=(p_1,\dots,p_m)$ and $Q=(q_1,\dots,q_m)$ be probability vectors on the same $m$ outcomes (nonnegative entries summing to $1$).
The **Kullback–Leibler divergence** of $Q$ from $P$ is
$$\mathrm{KL}(P\,\|\,Q)=\sum_{i=1}^m p_i\log\frac{p_i}{q_i},$$
with the natural logarithm, so KL is measured in **nats**.
Read it as an expectation under $P$: $\mathrm{KL}(P\|Q)=E_{i\sim P}[\log p_i-\log q_i]$, the average extra log-loss you pay for believing $Q$ when outcomes really come from $P$.

**Zero entries need conventions**, and they are not arbitrary:

* **$p_i=0$.** The term is $0\cdot\log(0/q_i)$, defined to be $0$. Justification: $\lim_{t\to0^+}t\log t=0$ (rewrite $t\log t=\frac{\log t}{1/t}$ and apply L'Hôpital's rule: $\frac{1/t}{-1/t^2}=-t\to0$). Outcomes that $P$ never produces cannot cost anything.
* **$q_i=0<p_i$.** The term is $p_i\log(p_i/0)=+\infty$, so $\mathrm{KL}(P\|Q)=+\infty$. A model that assigns probability zero to an outcome that actually happens is infinitely wrong under this measure.
* **$p_i=q_i=0$.** Covered by the first rule: the term is $0$.

In code, a naive `(p * torch.log(p / q)).sum()` returns `nan` for $p_i=0$ (because $0\cdot(-\infty)$ is `nan` in floating point) and hides the infinite case behind `inf`/`nan` mixtures.
A correct implementation masks: compute terms only where $p_i>0$, and return $+\infty$ explicitly if any $p_i>0$ has $q_i=0$.
This is exactly p08's contract.

**Checkpoint 1A.** Compute $\mathrm{KL}(P\|Q)$ for $P=(1/2,1/2,0)$ and $Q=(1/4,1/4,1/2)$.

**Checkpoint 1B.** What is $\mathrm{KL}(Q\|P)$ for the same pair, and which convention decides it?

**Collected answers.** $\frac12\log2+\frac12\log2+0=\log2\approx0.693147$ (the third term is $0$ by the $0\cdot\log0$ convention). $\mathrm{KL}(Q\|P)=+\infty$, because $q_3=1/2>0$ while $p_3=0$.

## 2. The tangent-line lemma and Jensen's inequality for $\log$

**Lemma (tangent line).** For every $x>0$, $\log x\le x-1$, with equality only at $x=1$.

*Proof.* Let $f(x)=x-1-\log x$ on $x>0$. Then $f'(x)=1-1/x$, which is negative on $(0,1)$ and positive on $(1,\infty)$. So $f$ decreases then increases, its unique minimum is at $x=1$, and $f(1)=0$. Hence $f(x)\ge0$ everywhere, with equality only at $x=1$. Geometrically, $y=x-1$ is the tangent line to the concave curve $y=\log x$ at $x=1$, and a concave curve lies below its tangent lines. $\square$

**Jensen's inequality for $\log$ (finite form).** Let $w_1,\dots,w_m\ge0$ with $\sum_i w_i=1$, and let $a_1,\dots,a_m>0$. Then
$$\sum_i w_i\log a_i\le\log\Big(\sum_i w_i a_i\Big),$$
with equality only when every $a_i$ with $w_i>0$ is the same number.

*Proof.* Let $A=\sum_i w_ia_i>0$. Apply the lemma to $x=a_i/A$: $\log a_i-\log A\le a_i/A-1$. Multiply by $w_i\ge0$ and sum: $\sum_iw_i\log a_i-\log A\le\frac{\sum_iw_ia_i}{A}-\sum_iw_i=1-1=0$. Equality forces equality in every lemma use with $w_i>0$, that is $a_i=A$. $\square$

**Expectation form.** The proof used only linearity, so it holds for any positive random variable $Y$ whose expectation exists: $E[\log Y]\le\log E[Y]$, with equality only if $Y$ is constant (with probability 1). Session 4 uses this form with $Y=p(x,z)/q(z\mid x)$ to derive the ELBO, and p14 uses the lemma directly. Book 1 F7 is not needed: this lemma is self-contained.

**Checkpoint 2A.** Verify Jensen numerically for $w=(1/2,1/2)$, $a=(1,4)$.

**Checkpoint 2B.** Where in the Jensen proof is $\sum_i w_i=1$ used?

**Collected answers.** Left side $\frac12(\log1+\log4)=\log2\approx0.693$; right side $\log(2.5)\approx0.916$; indeed $0.693\le0.916$. In the last step, $\frac{\sum_iw_ia_i}{A}-\sum_iw_i=1-1=0$; without it the right side would not be zero.

## 3. Gibbs' inequality and asymmetry, with worked numbers

**Gibbs' inequality.** $\mathrm{KL}(P\|Q)\ge0$, with equality exactly when $P=Q$.

*Proof via Jensen (strictly positive case).* Suppose every $p_i>0$ and $q_i>0$. Take weights $w_i=p_i$ and values $a_i=q_i/p_i$. Jensen gives $\sum_ip_i\log\frac{q_i}{p_i}\le\log\sum_ip_i\frac{q_i}{p_i}=\log\sum_iq_i=\log1=0$, so $-\mathrm{KL}(P\|Q)\le0$. Equality requires every $q_i/p_i$ to equal one common constant, and since both vectors sum to $1$ that constant is $1$. $\square$
When zeros are present, the $0\cdot\log0$ convention drops terms with $p_i=0$ and an infinite value is trivially nonnegative; writing that general case carefully, directly from the tangent-line lemma and with the equality condition, is p14.

**KL is not symmetric.** Take $P=(0.5,0.4,0.1)$ and $Q=(0.3,0.3,0.4)$:
$$\mathrm{KL}(P\|Q)=0.5\log\tfrac53+0.4\log\tfrac43+0.1\log\tfrac14=0.255413+0.115073-0.138629=0.231856,$$
$$\mathrm{KL}(Q\|P)=0.3\log\tfrac35+0.3\log\tfrac34+0.4\log4=-0.153248-0.086305+0.554518=0.314965.$$
The two directions differ, and Section 1's pair showed one direction finite ($\log2$) and the other infinite.
So KL is a **divergence**, not a distance: it is nonnegative and zero only at equality, but it is not symmetric (and it does not satisfy the triangle inequality either).
Always write the direction explicitly, and say which argument is the data distribution and which is the model.

**Checkpoint 3A.** In the Jensen proof, why can a single negative term such as $0.1\log\frac14$ appear without contradicting $\mathrm{KL}\ge0$?

**Checkpoint 3B.** If $\mathrm{KL}(P\|Q)=0$, what can you conclude about $P$ and $Q$?

**Collected answers.** Gibbs bounds the weighted sum, not each term; individual terms are negative wherever $q_i>p_i$, and they are outweighed by the positive terms. $P=Q$ exactly, by the equality condition.

In [ ]:
import math
import torch

def discrete_kl_demo(p, q):
    """Lesson demo: KL(P||Q) with the 0*log 0 convention and the infinite case."""
    total = 0.0
    for pi, qi in zip(p, q):
        if pi == 0.0:
            continue                      # 0 * log(0 / q) = 0
        if qi == 0.0:
            return math.inf               # q_i = 0 < p_i
        total += pi * math.log(pi / qi)
    return total

P, Q = (0.5, 0.4, 0.1), (0.3, 0.3, 0.4)
print("KL(P||Q) =", round(discrete_kl_demo(P, Q), 6), " KL(Q||P) =", round(discrete_kl_demo(Q, P), 6))
print("support example:", discrete_kl_demo((0.5, 0.5, 0.0), (0.25, 0.25, 0.5)),
      discrete_kl_demo((0.25, 0.25, 0.5), (0.5, 0.5, 0.0)))
naive = (torch.tensor([0.5, 0.5, 0.0]) * torch.log(torch.tensor([0.5, 0.5, 0.0]) / torch.tensor([0.25, 0.25, 0.5]))).sum()
print("naive vectorized value without a mask:", naive.item())     # nan: 0 * (-inf)

## 4. Forward versus reverse KL

When we **fit** a model $Q$ to a target $P$, the direction of KL changes what the fit rewards.

* **Forward KL, $\mathrm{KL}(P\|Q)$** (target first). The expectation is under $P$, and any outcome where $P$ has mass but $Q$ is tiny costs a lot. Minimizing it makes $Q$ **mass-covering**: $Q$ spreads out to cover every mode of $P$, even at the price of placing mass between modes. Maximum-likelihood fitting is forward KL in disguise.
* **Reverse KL, $\mathrm{KL}(Q\|P)$** (model first). The expectation is under $Q$, and placing $Q$-mass where $P$ is small costs a lot, while ignoring some of $P$'s mass is cheap. Minimizing it makes $Q$ **mode-seeking**: $Q$ locks onto one mode. The VAE's KL term, $\mathrm{KL}(q(z\mid x)\|p(z))$, has the model's distribution first.

**Worked numbers.** Target $P=(0.45,0.05,0.05,0.45)$ has two modes. Compare a broad model $Q_A=(0.25,0.25,0.25,0.25)$ with a one-mode model $Q_B=(0.85,0.05,0.05,0.05)$:

| candidate | forward $\mathrm{KL}(P\|Q)$ | reverse $\mathrm{KL}(Q\|P)$ |
|---|---:|---:|
| $Q_A$ broad | 0.368064 | 0.510826 |
| $Q_B$ one mode | 0.702556 | 0.430729 |

Forward KL prefers the broad $Q_A$ (mass-covering); reverse KL prefers $Q_B$ (mode-seeking).
Neither choice is "correct"; they answer different questions.
If you need samples that look like real modes, reverse-KL behaviour helps; if missing a mode is unacceptable, forward-KL behaviour helps.

**Checkpoint 4A.** Which direction becomes $+\infty$ if the model $Q$ assigns zero probability to an outcome the target $P$ produces?

**Checkpoint 4B.** Using the table, which candidate would a maximum-likelihood fit prefer, and why?

**Collected answers.** Forward KL, $\mathrm{KL}(P\|Q)$, because a term $p_i\log(p_i/0)$ appears. $Q_A$: maximum likelihood minimizes forward KL, and $0.368<0.703$.

## 5. Continuous KL and the univariate Gaussian closed form

For densities the sum becomes an expectation of the same log-ratio: $\mathrm{KL}(p\|q)=E_{x\sim p}[\log p(x)-\log q(x)]$.
Gibbs' inequality still holds (the expectation form of Jensen in Section 2 gives it the same way).

**Two expectation identities (Book 1 variance algebra).** If $x\sim N(m_1,s_1^2)$, then $E[(x-m_1)^2]=s_1^2$ by definition of variance, and for any constant $m$,
$$E[(x-m)^2]=E[(x-m_1)^2]+2(m_1-m)E[x-m_1]+(m_1-m)^2=s_1^2+(m_1-m)^2,$$
because $E[x-m_1]=0$.

**Closed form.** For $p=N(m_1,s_1^2)$ and $q=N(m_2,s_2^2)$, the log-densities are $\log p(x)=-\frac12\log(2\pi s_1^2)-\frac{(x-m_1)^2}{2s_1^2}$ and similarly for $q$. Subtract, take $E_{x\sim p}$, and apply the identities:
$$\mathrm{KL}(p\|q)=\log\frac{s_2}{s_1}+\frac{s_1^2+(m_1-m_2)^2}{2s_2^2}-\frac12 .$$
Sanity checks: it is $0$ when $m_1=m_2$ and $s_1=s_2$; it grows quadratically in the mean gap; it is not symmetric in $(s_1,s_2)$.
For example, $p=N(1,0.5^2)$ and $q=N(0,2^2)$ give $\log4+\frac{0.25+1}{8}-\frac12=1.386294+0.15625-0.5=1.042544$.

The special case $q=N(0,1)$ is the one the VAE needs; writing that derivation out in full from the definition, step by step, is p15.

**Checkpoint 5A.** Evaluate the closed form for $p=N(0,1)$, $q=N(0,4)$ (so $s_2=2$).

**Checkpoint 5B.** Which Book 1 identity turns $E_{x\sim p}[(x-m_2)^2]$ into $s_1^2+(m_1-m_2)^2$?

**Collected answers.** $\log2+\frac{1}{8}-\frac12=0.693147-0.375=0.318147$. The variance decomposition $E[(x-m)^2]=\operatorname{Var}(x)+(E[x]-m)^2$, from expanding the square and using $E[x-m_1]=0$.

## 6. Diagonal Gaussians, KL to $N(0,I)$, and log-variance

**Diagonal case.** If $p=N(\mu,\operatorname{diag}(\sigma^2))$ and $q=N(\nu,\operatorname{diag}(\tau^2))$ in $k$ dimensions, both log-densities are sums over coordinates (Session 1, Section 3), and expectation is linear, so the KL is the **sum of the $k$ univariate KLs**:
$$\mathrm{KL}(p\|q)=\sum_{j=1}^k\Big[\log\frac{\tau_j}{\sigma_j}+\frac{\sigma_j^2+(\mu_j-\nu_j)^2}{2\tau_j^2}-\frac12\Big].$$

**KL to the standard normal.** With $\nu=0$ and $\tau=1$, and using $\log\frac1{\sigma_j}=-\frac12\log\sigma_j^2$:
$$\mathrm{KL}\big(N(\mu,\operatorname{diag}\sigma^2)\,\|\,N(0,I)\big)=\frac12\sum_{j=1}^k\big(\mu_j^2+\sigma_j^2-\log\sigma_j^2-1\big).$$
Each term is nonnegative: $\mu_j^2\ge0$, and $\sigma_j^2-1-\log\sigma_j^2\ge0$ is the tangent-line lemma with $x=\sigma_j^2$.

**Log-variance parameterization.** Networks output an unconstrained real number `logvar` $=\log\sigma^2$ instead of $\sigma^2$, because any real `logvar` gives a positive variance `logvar.exp()` and a standard deviation `torch.exp(0.5 * logvar)`.
In code the KL becomes `0.5 * (mu**2 + logvar.exp() - logvar - 1).sum(dim=-1)`, shape `(B,)` for `mu, logvar` of shape `(B,k)`: summed over latent coordinates, one value per example.
Two traps: writing `sigma` where `sigma**2` belongs, and dropping the `-1` (which makes the KL of $N(0,I)$ to itself equal $k/2$ instead of $0$).
Both are exactly the wrong implementations that p09's checks must reject.

**Checkpoint 6A.** Compute the KL to $N(0,I_2)$ for $\mu=(0,0)$, $\log\sigma^2=(0,0)$.

**Checkpoint 6B.** A colleague's KL returns $1.0$ for $\mu=0$, `logvar` $=0$, $k=2$. Which term is missing?

**Collected answers.** $\frac12[(0+1-0-1)+(0+1-0-1)]=0$, as it must for identical distributions. The $-1$ term: without it each coordinate contributes $\frac12(0+1-0)=\frac12$, totalling $1$.

## 7. Worked laboratory: Gaussian-to-standard-normal KL

Take a 2-D diagonal Gaussian with $\mu=(1,-0.5)$ and $\sigma^2=(0.25,2)$, so `logvar` $=(\log0.25,\log2)=(-1.386294,0.693147)$.

**Coordinate 1.** $\frac12(1^2+0.25-\log0.25-1)=\frac12(0.25+1.3862944)=0.8181472$.

**Coordinate 2.** $\frac12((-0.5)^2+2-\log2-1)=\frac12(1.25-0.6931472)=0.2784264$.

**Total.** $\mathrm{KL}=0.8181472+0.2784264=1.0965736$ nats (printed as $1.096574$ below).

**Monte Carlo certificate.** KL is an expectation under $p$, so it can be estimated by sampling: draw $z^{(s)}=\mu+\sigma\odot\varepsilon^{(s)}$ with $\varepsilon^{(s)}\sim N(0,I)$ and average $\log p(z^{(s)})-\log N(z^{(s)};0,I)$.
With $200{,}000$ float64 draws from a generator seeded `20260927`, the estimate is $1.0942$ with standard error about $0.0028$, so the closed form sits within one standard error.
A sound comparison tolerance is several standard errors (here $0.02$ is about seven), not $10^{-6}$.

The first cell computes the closed form and the estimate; the second shows the two classic bugs failing the comparison.
This lab uses different numbers from p09, which asks you to implement `kl_to_standard_normal(mu, logvar)` against its own pinned Monte Carlo probe.

**Checkpoint 7A.** Recompute coordinate 2 if its variance were $1$ instead of $2$ (same mean).

**Checkpoint 7B.** Why is the Monte Carlo estimate itself never used as the reported KL in this unit?

**Collected answers.** $\frac12(0.25+1-0-1)=0.125$. It is random (it changes with the seed and has standard error about $0.003$ here), while the closed form is exact; the estimate is a certificate that the closed-form code is right, not a replacement.

In [ ]:
import math
import torch

mu = torch.tensor([1.0, -0.5], dtype=torch.float64)
logvar = torch.log(torch.tensor([0.25, 2.0], dtype=torch.float64))
closed = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0)
print("per-coordinate:", [round(v, 6) for v in closed.tolist()], " total:", round(closed.sum().item(), 6))

generator = torch.Generator().manual_seed(20260927)
eps = torch.randn(200_000, 2, generator=generator, dtype=torch.float64)
z = mu + torch.exp(0.5 * logvar) * eps
log_q = (-0.5 * (math.log(2 * math.pi) + logvar + (z - mu) ** 2 / logvar.exp())).sum(dim=1)
log_p = (-0.5 * (math.log(2 * math.pi) + z ** 2)).sum(dim=1)
ratio = log_q - log_p
estimate, std_error = ratio.mean().item(), (ratio.std() / math.sqrt(ratio.numel())).item()
print("Monte Carlo:", round(estimate, 4), "+/-", round(std_error, 4))
assert math.isclose(estimate, closed.sum().item(), abs_tol=0.02, rel_tol=0.0)

In [ ]:
import torch

mu = torch.tensor([1.0, -0.5], dtype=torch.float64)
logvar = torch.log(torch.tensor([0.25, 2.0], dtype=torch.float64))
correct = (0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0)).sum()
missing_minus_one = (0.5 * (mu ** 2 + logvar.exp() - logvar)).sum()
sigma_for_variance = (0.5 * (mu ** 2 + torch.exp(0.5 * logvar) - logvar - 1.0)).sum()
for name, value in [("correct", correct), ("drop -1", missing_minus_one), ("sigma for sigma^2", sigma_for_variance)]:
    print(f"{name:>18}: {value.item():.6f}")
assert not torch.isclose(missing_minus_one, correct, atol=0.02, rtol=0.0)
assert not torch.isclose(sigma_for_variance, correct, atol=0.02, rtol=0.0)

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *`nan` from zero probabilities.* `p * log(p / q)` gives `nan` at $p_i=0$. Fix: mask the $p_i>0$ terms; return `inf` when some $p_i>0$ meets $q_i=0$.
2. *Silent direction swap.* Computing $\mathrm{KL}(Q\|P)$ when the task asked for $\mathrm{KL}(P\|Q)$. Fix: write the expectation's distribution in a comment before coding.
3. *Wrong logarithm base.* `log2` gives bits; this unit's formulas and probes use natural log (nats).
4. *Variance versus standard deviation versus log-variance.* `logvar.exp()` is $\sigma^2$; `torch.exp(0.5 * logvar)` is $\sigma$. Mixing them breaks every closed form.
5. *Dropping the $-1$ or the $\frac12$.* Test at $\mu=0$, `logvar` $=0$, where the true KL is exactly $0$.
6. *Wrong reduction.* The VAE KL is summed over latent coordinates and then averaged over the batch; averaging over latents shrinks it by $k$.
7. *Exact comparison to Monte Carlo.* Use a tolerance of several standard errors.

**Exam connections.** Expect to compute a discrete KL by hand in both directions (p03), implement a guarded discrete KL and certify nonnegativity on many random pairs (p08), implement the closed-form diagonal KL with a Monte Carlo cross-check (p09), prove Gibbs' inequality from the tangent-line lemma (p14), and derive the Gaussian-to-standard-normal formula (p15). In p22 you read a KL trace to diagnose posterior collapse.

**Checkpoint 8A.** Which single test input exposes the 'drop the $-1$' bug, and why does that same input not expose a missing $\frac12$?

**Checkpoint 8B.** Name the practice that proves Gibbs' inequality directly from $\log x\le x-1$.

**Collected answers.** $\mu=0$, `logvar` $=0$: the correct KL is $0$, while dropping $-1$ gives $k/2$. A missing $\frac12$ doubles the value, and double of $0$ is still $0$, so you also need an input with nonzero KL, such as Section 7's example ($1.096574$ versus $2.193147$). p14.

## 9. Forward-only deeper topic: entropy, cross-entropy, and other divergences

Forward-only deeper topic.
KL splits as $\mathrm{KL}(P\|Q)=H(P,Q)-H(P)$, where $H(P)=-\sum_ip_i\log p_i$ is entropy and $H(P,Q)=-\sum_ip_i\log q_i$ is cross-entropy; that identity explains why minimizing cross-entropy over $Q$ is the same as minimizing forward KL.
Other families of divergences (Jensen–Shannon, $f$-divergences, Wasserstein distances) trade KL's asymmetry and infinite values for other properties; GANs in `B2-023-generative-models-diffusion` are often analysed with them.
Diffusion models in B2-023 are trained with a sum of Gaussian-to-Gaussian KL terms like the closed form of Section 6.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-022.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Does any B2-022 practice require entropy or Jensen–Shannon divergence?

**Checkpoint 9B.** Which closed form from this session reappears inside diffusion training objectives?

**Collected answers.** No; they are forward-only and not assessed. The KL between two diagonal Gaussians (Section 6), summed over coordinates.